In [1]:
from google.colab import drive

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from torch import nn
from torch.optim import Adam
from torch.nn import MSELoss
from torch import cat, float32, long, tensor
from torch.utils.data import Dataset, DataLoader

import tqdm


In [2]:
drive.mount('/content/drive')

train_data = pd.read_csv('/content/drive/My Drive/train.csv')

test_data = pd.read_csv('/content/drive/My Drive/test.csv')

train_data.head(10)

KeyboardInterrupt: 

In [ ]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=16):
        super(NCF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim * 2, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, user, item):
        user_emb = self.user_embedding(user)
        item_emb = self.item_embedding(item)
        x = cat([user_emb, item_emb], dim=-1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
import sys
!{sys.executable} -m pip uninstall sympy -y
!{sys.executable} -m pip install sympy

print("Sympy reinstalled. Please try running the training cell again.")

In [ ]:
drive.mount('/content/drive')

train_data = pd.read_csv('/content/drive/My Drive/train.csv')

test_data = pd.read_csv('/content/drive/My Drive/test.csv')

train_data.head(10)

### Training the NCF Model

In [ ]:
user_id_map = {id: i for i, id in enumerate(train_data['user'].unique())}
track_id_map = {id: i for i, id in enumerate(train_data['track'].unique())}

train_data['user_idx'] = train_data['user'].map(user_id_map)
train_data['track_idx'] = train_data['track'].map(track_id_map)

num_users = len(user_id_map)
num_items = len(track_id_map)

print(f"Number of unique users: {num_users}")
print(f"Number of unique items: {num_items}")

class MusicDataset(Dataset):
    def __init__(self, users, items, ratings):
        self.users = tensor(users.values, dtype=long)
        self.items = tensor(items.values, dtype=long)
        self.ratings = tensor(ratings.values, dtype=float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


train_dataset = MusicDataset(train_data['user_idx'], train_data['track_idx'], train_data['time'])
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

model = NCF(num_users, num_items)

criterion = MSELoss()
optimizer = Adam(model.parameters(), lr=0.001)

num_epochs = 5

for epoch in tqdm(range(num_epochs)):
    model.train()
    total_loss = 0
    for user, item, rating in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(user, item)
        loss = criterion(outputs.squeeze(), rating)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

print("Training complete.")